In [ ]:
import pandas as pd
import numpy as np
import time
import unicodedata


def remove_tildes(text):
    return unicodedata.normalize("NFD", text).encode("ascii", "ignore").decode("utf-8")


pd.set_option("display.max_rows", 100)

In [ ]:
cols_cie10 = ['Código', 'Descripción', 'Sección']
ruta = 'datasets/grd/CIE-10.xlsx'
cie10 = pd.read_excel(ruta, usecols=cols_cie10, dtype={col:'string' for col in cols_cie10})
cie10.rename(columns={
    'Código':'DIAGNOSTICO1',
    'Descripción':'DESCRIPCION',
    'Sección':'SECCION'
    }, inplace=True)
cie10.info()

In [ ]:
ruta = 'datasets/sae_multidimensional_2024.xlsx'
dtypes_pm = {
    'Porcentaje de personas en situacion de pobreza multidimensional 2024':'float32',
    'Código':'string'
}
cols_pm = list(dtypes_pm.keys())
pm = pd.read_excel(ruta, header=2, usecols=cols_pm, dtype=dtypes_pm)
pm['Código'] = pm['Código'].str.strip().str.zfill(5)
pm.rename(columns={
    'Porcentaje de personas en situacion de pobreza multidimensional 2024':'POB_MULTIDIM',
    'Código':'COD_COMUNA'
    }, inplace=True)
# pm['COMUNA_REGISTRADA'] = pm['COMUNA_REGISTRADA'].str.strip().str.upper().apply(remove_tildes)
pm.info()

In [ ]:
ruta = f"datasets/base establecimientos/Base de Establecimientos 2024.xlsx"
dtypes_estab = {'Código Vigente':'string',
        'Código Región':'str',
        'Nombre Región':'category',
        'Código Comuna':'str',
        'Nombre Comuna':'category',
        'Nombre Oficial':'string',
        'Tipo de Prestador Sistema de Salud': 'string',
        'Nivel de Complejidad': 'category',
        'LATITUD      [Grados decimales]':'float32',
        'LONGITUD [Grados decimales]':'float32',
        }
cols_estab = list(dtypes_estab.keys())
estab = pd.read_excel(ruta, header=1, usecols=cols_estab, dtype=dtypes_estab, 
                                na_values={"LATITUD      [Grados decimales]": "Pendiente",
                                "LONGITUD [Grados decimales]": "Pendiente"}, decimal=',')

estab['Código Región'] = estab['Código Región'].astype('string')
estab['Código Comuna'] = estab['Código Comuna'].astype('string')
estab['Código Vigente'] = estab['Código Vigente'].astype('string')
estab.rename(columns={
        "Código Vigente":"COD_ESTABLECIMIENTO",
        'Código Región':'COD_REGION',
        'Código Comuna':'COD_COMUNA',
        'Nombre Región':'REGION',
        'Nombre Comuna':'COMUNA',
        'Nombre Oficial':'ESTABLECIMIENTO',
        'Tipo de Prestador Sistema de Salud':'PRESTADOR',
        'Nivel de Complejidad':'COMPLEJIDAD_ESTAB',
        'LATITUD      [Grados decimales]':'LATITUD',
        'LONGITUD [Grados decimales]':'LONGITUD',
        }, inplace=True)

estab['COMUNA'] = estab['COMUNA'].str.strip().str.upper().apply(remove_tildes)
estab['COD_COMUNA'] = estab['COD_COMUNA'].str.strip().str.zfill(5)
estab['PRESTADOR'] = estab['PRESTADOR'].str.strip()
estab['ESTABLECIMIENTO'] = estab['ESTABLECIMIENTO'].str.strip()
estab = estab.dropna().reset_index(drop=True)
display(estab.shape, estab.info())

In [ ]:
ruta = 'datasets/Dotacion_Camas_Hospitalarias_2025.xlsx'
dtypes_camas = {
    'Código Establecimiento':'string',
    'Área Funcional':'string',
    **{year:'str' for year in range(2019,2025)}
}
cols_camas = list(dtypes_camas.keys())
camas = pd.read_excel(ruta, header=2, dtype=dtypes_camas, usecols=lambda col: col in cols_camas)

for year in range(2019,2025):
    camas.rename(columns={year:f'{year}'}, inplace=True)

camas.rename(columns={
    'Código Establecimiento':'COD_ESTABLECIMIENTO',
    'Área Funcional':'Area'
}, inplace=True)

camas['COD_ESTABLECIMIENTO'] = camas['COD_ESTABLECIMIENTO'].ffill()
camas = camas.dropna(
    subset= ['COD_ESTABLECIMIENTO']
    ).reset_index(drop=True)

totales = camas[camas['Area'] == 'Total'].copy()

camas_criticas = [
    'Área Cuidados Intensivos Adultos',
    'Área Cuidados Intermedios Adulto',
    'Área Cuidados Intensivos Pediátrica',
    'Área Cuidados Intermedios Pediátricos',
    'Área Neonatología Cuidados Intensivos',
    'Área Neonatología Cuidados Intermedios'
]
cols_camas_str = [f'{year}' for year in range(2019,2025)]
criticas = camas[camas['Area'].isin(camas_criticas)].copy()

totales = totales.drop(columns=['Area'])
criticas = criticas.drop(columns=['Area'])

for year in range(2019,2025):
    criticas[f'{year}'] = pd.to_numeric(criticas[f'{year}'], errors='coerce')
    totales[f'{year}'] = pd.to_numeric(totales[f'{year}'], errors='coerce')
    criticas[f'{year}'] = criticas[f'{year}'].fillna(0).astype('UInt16')
    totales[f'{year}'] = totales[f'{year}'].fillna(0).astype('UInt16')

criticas = pd.DataFrame(criticas.groupby('COD_ESTABLECIMIENTO', as_index=False).sum())
criticas = criticas.melt(
    id_vars='COD_ESTABLECIMIENTO',
    value_vars=cols_camas_str,
    var_name='YEAR',
    value_name='CAMAS_CRITICAS'
)

totales = totales.melt(
    id_vars='COD_ESTABLECIMIENTO',
    value_vars=cols_camas_str,
    var_name='YEAR',
    value_name='CAMAS_TOTALES'
)
camas = pd.merge(totales, criticas, on=['COD_ESTABLECIMIENTO','YEAR'], how='left')
# camas = camas[(camas['CAMAS_CRITICAS'] != 0) | (camas['CAMAS_TOTALES'] != 0)].reset_index(drop=True)
camas['CAMAS_CRITICAS'] = camas['CAMAS_CRITICAS'].fillna(0)
camas['YEAR'] = camas['YEAR'].astype('int32')
camas.info()

In [ ]:
ruta = 'datasets/Proyeccion_2002-2035_Comunas.xlsx'
dtypes_pob = {
    'Region':'string',
    'Comuna':'string',
    **{f'Poblacion {year}':'uint16' for year in range(2019,2025)}
}
cols_pob = list(dtypes_pob.keys())

poblacion = pd.read_excel(ruta, usecols=cols_pob, dtype=dtypes_pob)
poblacion = poblacion.melt(
    id_vars=['Region','Comuna'],
    value_vars=[f'Poblacion {year}' for year in range(2019,2025)],
    var_name='YEAR',
    value_name='Poblacion'
)

poblacion['YEAR'] = poblacion['YEAR'].str.replace('Poblacion ', '').astype('int32')

comunas = poblacion.groupby(
    by=['Region','Comuna','YEAR'],
    observed=True
    )['Poblacion'].sum().reset_index()
comunas.rename(columns={'Poblacion':'POB_COMUNA'}, inplace=True)

regiones = comunas.groupby(
    by=['Region','YEAR'],
    observed=True
    )['POB_COMUNA'].sum().reset_index()
regiones.rename(columns={'POB_COMUNA':'POB_REGION'}, inplace=True)

poblacion = pd.merge(
    comunas, 
    regiones, 
    on=['Region','YEAR'],
    how='left'
)
poblacion['Comuna'] = poblacion['Comuna'].str.strip().str.zfill(5)
poblacion.rename(columns={'Comuna':'COD_COMUNA', 'Region':'COD_REGION'}, inplace=True)
display(poblacion.shape, poblacion.info())

In [ ]:
dtypes_grd = {'COD_HOSPITAL':"string", 
        'CIP_ENCRIPTADO':'string',
        'SEXO':"category",
        'PREVISION':"category",
        'COMUNA':"category",
        'PROVINCIA':"category",
        'TIPOALTA':"category",
        'SERVICIO_SALUD':"category",
        'TIPO_PROCEDENCIA':"category",
        'TIPO_INGRESO':"category",
        **{f'DIAGNOSTICO{i}':"string" for i in range(1,11)},
        **{f'PROCEDIMIENTO{i}':"string" for i in range(1,31)},
        'IR_29301_COD_GRD':'string',
        'IR_29301_PESO':'string',
        'IR_29301_SEVERIDAD':"category",
        'IR_29301_MORTALIDAD':"category"
        }

target_cols = list(dtypes_grd.keys())
cols_fechas = ['FECHA_NACIMIENTO', 'FECHA_INGRESO', 'FECHAALTA']
target_cols.extend(cols_fechas)
config_carga = {
        "sep": "|",
        "parse_dates": cols_fechas,
        "low_memory":False,
        "usecols": target_cols,
        "on_bad_lines": "skip",
        "dtype": dtypes_grd,
}

In [ ]:
ENCODINGS_TO_TRY = ['utf-8-sig', 'utf-16', 'utf-8', 'cp1252', 'latin1']

def detect_encoding(filepath):
    for enc in ENCODINGS_TO_TRY:
        try:
            with open(filepath, 'r', encoding=enc) as f:
                f.read(8192)
            return enc
        except (UnicodeDecodeError, UnicodeError):
            continue
    return 'latin1'  # last resort, never crashes


grds = {}
for year in range(2019,2025):
    print(f'opening grd{year}...')
    params = config_carga.copy()
    cols = target_cols.copy()
    ruta = f"datasets/grd/GRD_PUBLICO_{year}.txt"
    encoding = detect_encoding(ruta)
    inicio = time.time()
    if year == 2024:
        cols[1] = 'ID_BENEFICIARIO'
        params['usecols'] = cols
    grds[f'grd{year}'] = pd.read_csv(ruta, encoding = encoding, **params)
    final = time.time()
    print(f'loaded grd{year} on {final-inicio:.3f}s\n')

In [ ]:
# se estandariza la fecha de 2023, que tiene ciertas columnas con dias primero
grds['grd2023']['FECHAALTA'] = pd.to_datetime(grds['grd2023']['FECHAALTA'], dayfirst=True, errors='coerce')
grds['grd2023']['FECHA_INGRESO'] = pd.to_datetime(grds['grd2023']['FECHA_INGRESO'], dayfirst=True, errors='coerce')
grds['grd2024'].rename(columns={"ID_BENEFICIARIO":"CIP_ENCRIPTADO"}, inplace=True)
grd = pd.concat(grds.values(), ignore_index=True)
# se pasan las columnas a dtype datetime64
for col in cols_fechas:
    grd[col] = pd.to_datetime(grd[col], errors='coerce')

In [ ]:
# se lee el rut encriptado como dtype string, para que no se considere un
# numero y aparezca en las estadisticas erroneamente
grd['CIP_ENCRIPTADO'] = grd['CIP_ENCRIPTADO'].astype('string')

# se cambia la coma por punto para interpretar como float
grd['IR_29301_PESO'] = grd.IR_29301_PESO.str.replace(',','.', regex=False)
# los que NO tienen punto
dec_coma = ~grd['IR_29301_PESO'].str.contains('.', regex=False)
# todos como float
floats = pd.to_numeric(grd['IR_29301_PESO'], errors='coerce')
# grd['IR_29301_PESO'] = grd.IR_29301_PESO.str.replace('.','')
powers = np.maximum(grd['IR_29301_PESO'].str.len() - 1,1)
# se divide solo donde no hay punto, por error de tipeo
grd['IR_29301_PESO'] = np.where(dec_coma, floats / (10 ** powers), floats)

# columnas categoricas, que guardan valores unicos, o categorias de pandas
categ_cols = ['SEXO', 'PREVISION', 'TIPO_PROCEDENCIA', 'TIPO_INGRESO', 'IR_29301_SEVERIDAD', 'IR_29301_MORTALIDAD',
            'TIPOALTA', 'PROVINCIA', 'COMUNA','SERVICIO_SALUD']

# se eliminan nulos tras estandarizar la mayoria de columnas
cols_ignore = [f'PROCEDIMIENTO{i}' for i in range(1,31)] + [f'DIAGNOSTICO{i}' for i in range(2,11)]
grd = grd.dropna(subset=[col for col in grd.columns if col not in cols_ignore])
# se transforman las columnas anteriores a categoricas
grd = grd.astype({col: 'category' for col in categ_cols})
# se reescriben ciertos nombres de columna para calzar con los merges proximos
grd.rename(columns={'COMUNA':'COMUNA_REGISTRADA', 'COD_HOSPITAL':'COD_ESTABLECIMIENTO'}, inplace=True)
# se filtran datos para que solo sean con fecha ingreso >= 2019
grd['YEAR'] = grd['FECHA_INGRESO'].dt.year
grd['YEAR'] = grd['YEAR'].astype('int32')
grd = grd[grd['YEAR'] >= 2019]
grd.reset_index(drop=True, inplace=True)

# se estandarizan las comunas para que la registrada y del establecimiento
# esten escritas de la misma manera y puedan compararse
grd['COMUNA_REGISTRADA'] = grd['COMUNA_REGISTRADA'].str.strip().str.upper().apply(remove_tildes)
grd['COMUNA_REGISTRADA'] = grd['COMUNA_REGISTRADA'].str.replace('COIHAIQUE','COYHAIQUE')
grd['COMUNA_REGISTRADA'] = grd['COMUNA_REGISTRADA'].str.replace('AISEN','AYSEN')
grd['COMUNA_REGISTRADA'] = grd['COMUNA_REGISTRADA'].str.replace('CON CON','CONCON')
grd['COMUNA_REGISTRADA'] = grd['COMUNA_REGISTRADA'].str.replace('O"HIGGINS','OHIGGINS')

# se crean variables clave, edad y dias estancia
grd['EDAD'] = ((grd['FECHA_INGRESO'] - grd['FECHA_NACIMIENTO']).dt.days / 365.25).round(1)
grd['DIAS_ESTANCIA'] = (grd['FECHAALTA'] - grd['FECHA_INGRESO']).dt.days
grd['IR_29301_SEVERIDAD'] = grd.IR_29301_SEVERIDAD.astype('string').astype('category')
diag_cols = [f'DIAGNOSTICO{i}' for i in range(2,11)]
for col in diag_cols:
    grd[col] = grd[col].astype('string')
    grd[col].replace({'DESCONOCIDO':np.nan})
    grd[col] = grd[col].astype('category')
grd['N_COMORBILIDADES'] = grd[[f'DIAGNOSTICO{i}' for i in range(2,11)]].count(axis=1)
grd['N_PROCEDIMIENTOS'] = grd[[f'PROCEDIMIENTO{i}' for i in range(1,31)]].count(axis=1)

In [ ]:
# ================================================================
# ÍNDICE DE COMORBILIDAD DE CHARLSON — IMPLEMENTACIÓN PROPIA
# Referencia: Quan et al. (2011) Med Care, 49(6), 612–617.
#             Charlson et al. (1987) J Chronic Dis, 40(5), 373–383.
# ================================================================

# Mapeamos prefijos ICD-10 (3 caracteres) a la categoría Charlson.
# Cada categoría tiene un peso según el modelo original de Charlson
# actualizado por Quan et al. (2011).

CHARLSON_ICD10 = {
    # ── Peso 1 ──────────────────────────────────────────────
    # Infarto agudo de miocardio
    'I21': ('MI', 1), 'I22': ('MI', 1), 'I25': ('MI', 1),
    # Insuficiencia cardíaca congestiva
    'I50': ('CHF', 1),
    # Enfermedad vascular periférica
    'I70': ('PVD', 1), 'I71': ('PVD', 1), 'I73': ('PVD', 1),
    'I77': ('PVD', 1), 'I79': ('PVD', 1),
    # Enfermedad cerebrovascular
    'G45': ('CVD', 1), 'G46': ('CVD', 1),
    'I60': ('CVD', 1), 'I61': ('CVD', 1), 'I62': ('CVD', 1),
    'I63': ('CVD', 1), 'I64': ('CVD', 1), 'I65': ('CVD', 1),
    'I66': ('CVD', 1), 'I67': ('CVD', 1), 'I68': ('CVD', 1),
    'I69': ('CVD', 1),
    # Demencia
    'F00': ('DEM', 1), 'F01': ('DEM', 1), 'F02': ('DEM', 1),
    'F03': ('DEM', 1), 'G30': ('DEM', 1), 'G31': ('DEM', 1),
    # Enfermedad pulmonar crónica (EPOC)
    'J40': ('COPD', 1), 'J41': ('COPD', 1), 'J42': ('COPD', 1),
    'J43': ('COPD', 1), 'J44': ('COPD', 1), 'J45': ('COPD', 1),
    'J46': ('COPD', 1), 'J47': ('COPD', 1), 'J60': ('COPD', 1),
    'J61': ('COPD', 1), 'J62': ('COPD', 1), 'J63': ('COPD', 1),
    'J64': ('COPD', 1), 'J65': ('COPD', 1), 'J66': ('COPD', 1),
    'J67': ('COPD', 1),
    # Enfermedad del tejido conectivo
    'M05': ('RHD', 1), 'M06': ('RHD', 1), 'M32': ('RHD', 1),
    'M33': ('RHD', 1), 'M34': ('RHD', 1), 'M35': ('RHD', 1),
    # Úlcera péptica
    'K25': ('PUD', 1), 'K26': ('PUD', 1), 'K27': ('PUD', 1),
    'K28': ('PUD', 1),
    # Enfermedad hepática leve
    'B18': ('MLD', 1), 'K70': ('MLD', 1), 'K71': ('MLD', 1),
    'K73': ('MLD', 1), 'K74': ('MLD', 1), 'K76': ('MLD', 1),
    # Diabetes sin complicación
    'E10': ('DM', 1), 'E11': ('DM', 1), 'E12': ('DM', 1),
    'E13': ('DM', 1), 'E14': ('DM', 1),

    # ── Peso 2 ──────────────────────────────────────────────
    # Hemiplejia
    'G04': ('HEMI', 2), 'G81': ('HEMI', 2), 'G82': ('HEMI', 2),
    'G83': ('HEMI', 2),
    # Enfermedad renal moderada/severa
    'N03': ('REN', 2), 'N05': ('REN', 2), 'N18': ('REN', 2),
    'N19': ('REN', 2), 'N25': ('REN', 2),
    # Diabetes con complicación crónica
    'E102': ('DMC', 2), 'E112': ('DMC', 2), 'E122': ('DMC', 2),
    'E132': ('DMC', 2), 'E142': ('DMC', 2),
    # Tumor sólido sin metástasis
    'C00': ('TUM', 2), 'C01': ('TUM', 2), 'C02': ('TUM', 2),
    'C03': ('TUM', 2), 'C04': ('TUM', 2), 'C05': ('TUM', 2),
    'C06': ('TUM', 2), 'C07': ('TUM', 2), 'C08': ('TUM', 2),
    'C09': ('TUM', 2), 'C10': ('TUM', 2), 'C11': ('TUM', 2),
    'C12': ('TUM', 2), 'C13': ('TUM', 2), 'C14': ('TUM', 2),
    'C15': ('TUM', 2), 'C16': ('TUM', 2), 'C17': ('TUM', 2),
    'C18': ('TUM', 2), 'C19': ('TUM', 2), 'C20': ('TUM', 2),
    'C21': ('TUM', 2), 'C22': ('TUM', 2), 'C23': ('TUM', 2),
    'C24': ('TUM', 2), 'C25': ('TUM', 2), 'C26': ('TUM', 2),
    'C30': ('TUM', 2), 'C31': ('TUM', 2), 'C32': ('TUM', 2),
    'C33': ('TUM', 2), 'C34': ('TUM', 2), 'C37': ('TUM', 2),
    'C38': ('TUM', 2), 'C39': ('TUM', 2), 'C40': ('TUM', 2),
    'C41': ('TUM', 2), 'C43': ('TUM', 2), 'C45': ('TUM', 2),
    'C46': ('TUM', 2), 'C47': ('TUM', 2), 'C48': ('TUM', 2),
    'C49': ('TUM', 2), 'C50': ('TUM', 2), 'C51': ('TUM', 2),
    'C52': ('TUM', 2), 'C53': ('TUM', 2), 'C54': ('TUM', 2),
    'C55': ('TUM', 2), 'C56': ('TUM', 2), 'C57': ('TUM', 2),
    'C58': ('TUM', 2), 'C60': ('TUM', 2), 'C61': ('TUM', 2),
    'C62': ('TUM', 2), 'C63': ('TUM', 2), 'C64': ('TUM', 2),
    'C65': ('TUM', 2), 'C66': ('TUM', 2), 'C67': ('TUM', 2),
    'C68': ('TUM', 2), 'C69': ('TUM', 2), 'C70': ('TUM', 2),
    'C71': ('TUM', 2), 'C72': ('TUM', 2), 'C73': ('TUM', 2),
    'C74': ('TUM', 2), 'C75': ('TUM', 2), 'C76': ('TUM', 2),
    'C97': ('TUM', 2),
    # Leucemia
    'C91': ('LEU', 2), 'C92': ('LEU', 2), 'C93': ('LEU', 2),
    'C94': ('LEU', 2), 'C95': ('LEU', 2),
    # Linfoma
    'C81': ('LYM', 2), 'C82': ('LYM', 2), 'C83': ('LYM', 2),
    'C84': ('LYM', 2), 'C85': ('LYM', 2), 'C88': ('LYM', 2),
    'C90': ('LYM', 2),

    # ── Peso 3 ──────────────────────────────────────────────
    # Enfermedad hepática moderada/severa
    'K70': ('MSLD', 3), 'K72': ('MSLD', 3), 'K76': ('MSLD', 3),

    # ── Peso 6 ──────────────────────────────────────────────
    # Tumor sólido con metástasis
    'C77': ('METS', 6), 'C78': ('METS', 6),
    'C79': ('METS', 6), 'C80': ('METS', 6),
    # VIH / SIDA
    'B20': ('AIDS', 6), 'B21': ('AIDS', 6),
    'B22': ('AIDS', 6), 'B24': ('AIDS', 6),
}

print(f"Mapa Charlson cargado: {len(CHARLSON_ICD10)} prefijos ICD-10 mapeados")
print(f"Categorías únicas: {len(set(v[0] for v in CHARLSON_ICD10.values()))}")
indice_charlson = {key:val[1] for key,val in CHARLSON_ICD10.items()}
results = []
for col in diag_cols:
    result = grd[col].str[:3].map(indice_charlson).fillna(0)
    results.append(result)
grd['INDICE_CHARLSON'] = pd.concat(results, axis=1).sum(axis=1)

In [ ]:
grd.to_parquet('grd_19-24.parquet')

In [ ]:
# import pandas as pd
# grd = pd.read_parquet('grd_19-24.parquet')

In [ ]:
# se consideran casos de peso > 2 y severidad 2 o 3, quedando con 6.4% del dataset
grd = grd[grd['IR_29301_PESO'] > 2]
grd = grd[(grd['IR_29301_SEVERIDAD'] == '2') | (grd['IR_29301_SEVERIDAD'] == '3')]
grd = grd.reset_index(drop=True)

In [ ]:
grd.to_parquet('grd_ac_19-24.parquet')

In [ ]:
# import pandas as pd
# grd = pd.read_parquet('grd_ac_19-24.parquet')

In [ ]:
# se realizan todos los merge para los que se preparo la base de grd
# merge con cie10.
grd = pd.merge(grd, cie10, on='DIAGNOSTICO1', how='left')
# merge con base establecimientos
grd = pd.merge(grd, estab, on='COD_ESTABLECIMIENTO',how='left')
# merge con proyeccion poblacion del INE
grd = pd.merge(grd, poblacion, on=['COD_REGION','COD_COMUNA','YEAR'], how='left')
# merge con camas hospitalarias por establecimiento
grd = pd.merge(grd, camas, on=['COD_ESTABLECIMIENTO','YEAR'], how='left')
# merge con indice pobreza multidimensional
grd = pd.merge(grd, pm, on='COD_COMUNA', how='left')
grd.rename(columns={'COMUNA':'COMUNA_ESTAB', 'POB_MULTIDIM':'POB_MULT_ESTAB'}, inplace=True)
grd['POB_MULT_ESTAB'] = grd['POB_MULT_ESTAB'].fillna(grd['POB_MULT_ESTAB'].mean())
grd['COMUNA_CON_ESTAB_AC'] = grd['COMUNA_REGISTRADA'].isin(grd['COMUNA_ESTAB']).astype(int)
# grd['TIENE_ESTAB_AC'] = grd['COMUNA_CON_ESTAB'].map({True:'Si',False:'No'})
grd["FALLECIDO"] = (grd["TIPOALTA"] == "FALLECIDO").astype(int)
grd['CRITICAS_PROP'] = grd['CAMAS_CRITICAS'] / grd['CAMAS_TOTALES']
estab_unicos_comuna = grd.drop_duplicates(subset=['COMUNA_ESTAB','COD_ESTABLECIMIENTO','YEAR'])
comunas_unique = estab_unicos_comuna.groupby(['COMUNA_ESTAB','YEAR'],observed=True)
camas_comuna = comunas_unique['CAMAS_TOTALES'].sum().reset_index(name='CAMAS_COMUNA')
criticas_comuna = comunas_unique['CAMAS_CRITICAS'].sum().reset_index(name='CRITICAS_COMUNA')
grd = grd.merge(camas_comuna, on=['COMUNA_ESTAB','YEAR'], how='left')
grd = grd.merge(criticas_comuna, on=['COMUNA_ESTAB','YEAR'], how='left')
grd['CAMAS_1000HAB'] = grd['CAMAS_COMUNA']/grd['POB_COMUNA'] * 1000
grd['CRITICAS_1000HAB'] = grd['CRITICAS_COMUNA']/grd['POB_COMUNA'] * 1000
tasa_mortalidad_comuna = grd.groupby('COMUNA_ESTAB', observed=True)['FALLECIDO'].mean()
grd['TASA_MORTALIDAD_COMUNA'] = grd['COMUNA_ESTAB'].map(tasa_mortalidad_comuna)


In [ ]:
# se transforman los dtypes de las nuevas columnas de poblacion
grd['POB_COMUNA'] = grd['POB_COMUNA'].astype('int32')
grd['POB_REGION'] = grd['POB_REGION'].astype('int32')
# se dropean nulos tras merge
grd = grd = grd.dropna(subset=[col for col in grd.columns if col not in cols_ignore]).reset_index(drop=True)
categ_cols = ['COD_ESTABLECIMIENTO', 'COD_COMUNA','COD_REGION','YEAR','COMUNA_REGISTRADA']
for col in categ_cols:
    grd[col] = grd[col].astype('category')
display(grd.shape, grd.info())

In [ ]:
grd.to_parquet("grd_procesado.parquet")